In [14]:
# TODO 1. Fragen und Antworten übersetzen
# 2. max. Punkte für Antworten vergeben
# 3. Schlüsselbegriffe herausfiltern
# 4. Klassifizierung (Abfrage, Hauptfrage, Ergänzungsfrage)

In [ ]:
# Datei einlesen

import pandas as pd
from llm_handler import LLMHandler

path = "../raw/DataScienceBasics_QandA - Sheet1.csv"
df = pd.read_csv(path)
df.head()

,Id,Question,Answer
0,1,What is data science?,Data science is an interdisciplinary field tha...
1,2,What are the key steps in the data science pro...,The key steps typically include problem defini...
2,3,What is the difference between supervised and ...,Supervised learning involves training a model ...
3,4,Explain the bias-variance tradeoff.,The bias-variance tradeoff is the balance betw...
4,5,What is feature engineering?,Feature engineering is the process of selectin...


In [16]:
# Prompt schreiben


def construct_prompt(question, answer):
    return f"""Du bist Dozent für Data Science sprichst sowohl Englischen als auch Deutsch wie ein Muttersprachler.
    Führe folgende Schritte aus:
    1. Übersetze die englische Frage {question} und die dazugehörige Antwort {answer} bestmöglich ins Deutsche.
        Halte dich dabei kurz und erfinde nichts dazu. 
    
    2. Entscheide wieviele Punkt die jeweilige Antwort in einer Prüfung maximal wert ist.
        Bei der Bewertung orientiere dich an folgender Struktur:
        - 1 bis 3 Punkte: Basiswissen
        - 4 bis 5 Punkte: einfaches Verständnis
        - 6 bis 8 Punkte: erweiterte Anwendung
        - 9 bis 10 Punkte: Expertenlevel
    
    3. Überlege welche Schlüsselbegriff die Antwort enthält, welche der Student unbedingt verwenden sollte
      und für die man bei Nichtnennung im Zweifel gezielt eine Rückfrage generieren könnte.

    4. Klassifiziere die Frage. Nutze dabei folgende Klassifizierung:
        - "Hauptfrage": Die Frage ergibt im Kontext Data Science eigenständig Sinn und kann ohne Abhängigkeiten oder Bezüge zu einer vorausgegangenen Frage gestellt werden.
        - "Vertiefung": Die Frage geht spezifisch auf Teilaspekte einer eher übergeordneten Frage ein und sollte daher eher in Zusammenhang mit einer vorausgegangenen Frage gestellt werden.
        - "Aufzählung": Die Frage dient primär dem einfachen Abfragen und Nennen von Aufzählungen oder Beispielen.
        - "Sonstige": Die Frage passt in keine der 3 vorher genannten Kategorien.
    
    5. Gib deine Antwort im angegebenen Antwortformat zurück.

    <Antwortformat>
    ```json
    {{
        "question": "<Die übersetzte Frage vom Englischen ins Deutsche>",
        "answer": "<Die übersetzte Antwort vom Englischen ins Deutsche>",
        "max_points": number,
        "keywords": "<keyword1; keyword2; keyword3;...>",
        "classification": "Hauptfrage | Vertiefung | Aufzählung | Sonstige"
    }}```
    """

In [17]:
# LLM aufrufen und in Dataframe speichern

llm = LLMHandler()
result_list = []

for index, row in df.iterrows():
    prompt = construct_prompt(row["Question"], row["Answer"])
    answer = llm.call_llm(prompt)
    answer["original_question"] = row["Question"]
    answer["original_answer"] = row["Answer"]
    result_list.append(answer)

process_data = pd.DataFrame(result_list)

In [18]:
# Daten CSV speichern

process_data.to_csv("generated_q_and_a_new.csv")

process_data.head()

,question,answer,max_points,keywords,classification,original_question,original_answer
0,Was ist Data Science?,"Data Science ist ein interdisziplinäres Feld, ...",6,interdisziplinär; wissenschaftliche Methoden; ...,Hauptfrage,What is data science?,Data science is an interdisciplinary field tha...
1,Welche sind die Schlüsselschritte im Data-Scie...,Die Schlüsselschritte umfassen typischerweise ...,4,Problemdefinition; Datensammlung; Datenbereini...,Aufzählung,What are the key steps in the data science pro...,The key steps typically include problem defini...
2,Was ist der Unterschied zwischen überwachtem u...,Überwachtes Lernen beinhaltet das Trainieren e...,6,überwachtes Lernen; unüberwachtes Lernen; gela...,Hauptfrage,What is the difference between supervised and ...,Supervised learning involves training a model ...
3,Erkläre das Bias-Variance-Trade-off.,Das Bias-Variance-Trade-off ist der Kompromiss...,6,Bias-Variance-Trade-off; Bias; Variance; Unter...,Hauptfrage,Explain the bias-variance tradeoff.,The bias-variance tradeoff is the balance betw...
4,Was ist Feature Engineering?,"Feature Engineering ist der Prozess, Features ...",6,Feature Engineering; Rohdaten; Auswahl; Transf...,Hauptfrage,What is feature engineering?,Feature engineering is the process of selectin...
